In [ ]:
#creates a list of station coordinates in X,Y,Z
import os
import numpy as np
from pyproj import Proj, Transformer
import csv

# Define the directory containing the station list files
station_list_dir = r"../prec/data_byEQ"  # Use raw string for Windows paths

# Define the output directory and file path
output_dir = r"../out"
output_csv = os.path.join(output_dir, "station_coordinates.csv")

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Define the coordinate transformation from WGS84 (latitude/longitude) to ECEF (XYZ)
transformer = Transformer.from_proj(
    Proj(proj='latlong', datum='WGS84'),
    Proj(proj='geocent', datum='WGS84'),
    always_xy=True  # Longitude first for consistency with Proj
)

def batch_latlon_to_xyz(latitudes, longitudes, altitudes=None):
    if altitudes is None:
        altitudes = np.zeros_like(latitudes)

    # Perform the transformation for all points in one go
    X, Y, Z = transformer.transform(longitudes, latitudes, altitudes)
    return X, Y, Z

# List to collect results in the format [Name, X, Y, Z]
results = []
processed_files = []  # List to keep track of processed files

# Search for all station_list.txt files in the directory and subdirectories
for root, dirs, files in os.walk(station_list_dir):
    for filename in files:
        # Look for .txt files with the expected name pattern
        if filename.lower() == "station_list.txt":
            filepath = os.path.join(root, filename)
            processed_files.append(filepath)  # Add full path of the file to the processed list

            # Open and process the file
            latitudes = []
            longitudes = []
            names = []

            with open(filepath, "r") as file:
                for line in file:
                    # Split the line into Name, Longitude, Latitude
                    parts = line.split()
                    if len(parts) == 3:
                        try:
                            name = parts[0]  # First part is the Name
                            longitude = float(parts[1])  # Second part is the Longitude
                            latitude = float(parts[2])  # Third part is the Latitude
                            names.append(name)
                            longitudes.append(longitude)
                            latitudes.append(latitude)
                        except ValueError:
                            # Skip lines that do not contain valid numbers
                            continue

            # Only process files that have valid lat/lon data
            if latitudes and longitudes and names:
                # Convert latitudes and longitudes to numpy arrays for vectorized operations
                latitudes = np.array(latitudes)
                longitudes = np.array(longitudes)

                # Convert all lat/lon pairs in this file to XYZ
                X, Y, Z = batch_latlon_to_xyz(latitudes, longitudes)

                # Append each result to the results list with the Name and XYZ coordinates
                for name, x, y, z in zip(names, X, Y, Z):
                    results.append([name, x, y, z])

# Print the list of processed input files
print("Processed station_list.txt files:")
for file in processed_files:
    print(file)

# Print the results
print("\nName\t\tX\t\tY\t\tZ")
for result in results:
    print(f"{result[0]}\t{result[1]:.2f}\t{result[2]:.2f}\t{result[3]:.2f}")

# Save results to a CSV file in the specified output directory
with open(output_csv, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Name", "X", "Y", "Z"])  # Header
    writer.writerows(results)

print(f"\nResults saved to {output_csv}")

In [35]:
#check all files have been input files have been processed
import os

# Define the parent directory where the output files were saved
output_directory = r"../prec/data_byEQ"  # Update this path if needed

# Count the number of .gta files (or the appropriate output extension)
output_files = []
for root, dirs, files in os.walk(output_directory):
    for file in files:
        if file.endswith("_GTS_CME.gta"):  # Replace with correct suffix if different
            output_files.append(os.path.join(root, file))

# Display the number of processed files
print(f"Number of processed files: {len(output_files)}")

Number of processed files: 3854


In [33]:
#creates output .gta files for all input files in the format for GTS_CME
import os
import pandas as pd
import re 
from datetime import datetime as dt
import dateutils as dateutils
import numpy as np
import glob 

# Set the main directory
parent_directory = r"../prec/data_byEQ"  # Update with actual path

# Columns to remove from each .dat file (0-indexed)
columns_to_remove = [0, 2, 3, 4, 5, 6]

# Path to the CSV file with station coordinates
station_coordinates_csv = r"../output/station_coordinates.csv"  # Path where the uploaded file is located

def datenum(d):
    return 366 + d.toordinal() + (d - dt.fromordinal(d.toordinal())).total_seconds()/(24*60*60) 
    
# Load station coordinates from CSV
try:
    # Read station coordinates with appropriate column names
    station_coords_df = pd.read_csv(station_coordinates_csv)

    # Assuming columns: Name, X, Y, Z; strip ".dat" if needed
    station_coordinates = {
        row['Name']: (row['X'], row['Y'], row['Z'])
        for _, row in station_coords_df.iterrows()
    }

except FileNotFoundError:
    print(f"Error: {station_coordinates_csv} not found.")
    raise
except KeyError as e:
    print(f"Error: Expected column not found in the CSV. Missing column: {e}")
    raise

# List all .dat files in the directory and subdirectories
dat_files = []
for root, dirs, files in os.walk(parent_directory):
    for file in files:
        if file.endswith(".dat"):
            dat_files.append(os.path.join(root, file))

# Process each .dat file and apply modifications
station_counter = {}  # To track duplicate station names
processed_rows = {}  # Dictionary to track processed rows for each station

for file_path in dat_files:
    # Load the .dat file
    data = pd.read_csv(file_path, sep=r'\s+', header=None)
    
    # Extract the station name from the filename
    filename = os.path.basename(file_path)
    station_name = filename.split("_")[1].split('.')[0]  # Assuming station name is between TS_ and .dat
    
    # Remove any "(number)" from station names
    station_name = re.sub(r'\(\d+\)', '', station_name).strip()
    
    # Check if the station exists in the station coordinates dictionary
    if station_name in station_coordinates:
        # Retrieve XYZ coordinates (don't swap these coordinates)
        x, y, z = station_coordinates[station_name]
    else:
        print(f"Warning: Station '{station_name}' not found in station coordinates.")
        continue  # Skip files with no matching station coordinates

    # Keep track of duplicate station names
    if station_name not in station_counter:
        station_counter[station_name] = 1
        output_filename = f"{station_name}_GTS_CME.gta"  # First occurrence, no suffix
    else:
        station_counter[station_name] += 1
        output_filename = f"{station_name}_GTS_CME.gta"  # Same filename, no duplicates
    
    # Define output file path to be in the same directory as the input file
    output_file_path = os.path.join(os.path.dirname(file_path), output_filename)

    # Initialize a set for the current station if not already present in the processed_rows dictionary
    if station_name not in processed_rows:
        processed_rows[station_name] = set()

    dn_array = []
    for i,mjd in enumerate(data.iloc[:,1]): 
        date = dateutils.mjd_to_date(mjd)
        dn = datenum(date)+int(data.iloc[i,6])/(3600*24)
        dn_array.append(dn)

    data[1] = dn_array
    
    # Convert columns 7, 8, and 9 from meters to millimeters
    data[7] *= 1000  # Convert to mm
    data[8] *= 1000  # Convert to mm
    data[9] *= 1000  # Convert to mm
    
    # Remove the specified columns
    data = data.drop(columns=columns_to_remove)
    
    # Swap the 2nd and 3rd columns (columns 1 and 2) as needed
    reordered_data = data.copy()
    reordered_data.iloc[:, [1, 2]] = reordered_data.iloc[:, [2, 1]]

    # Write to .gta file in the specified format
    with open(output_file_path, 'w') as f:
        # Write header with station information in the correct order
        f.write(f"#Site: {station_name}\n")
        f.write(f"#X: {x}\n")  # Station X (longitude)
        f.write(f"#Y: {y}\n")  # Station Y (latitude)
        f.write(f"#Z: {z}\n")  # Station Z (elevation)
        f.write("# T N E U\n")  # Custom header for data columns

        # Write the reordered data rows
        for _, row in reordered_data.iterrows():
            row_str = " ".join(f"{val:.4f}" for val in row)
            f.write(f"{row_str}\n")